In [0]:
try:
    import polars as pl
except ModuleNotFoundError:
    !pip install polars
    import polars as pl

try:
    import kagglehub
except ModuleNotFoundError:
    !pip install kagglehub
    import kagglehub

import os

# Funções:

In [0]:
def check_primary_key(df:pl.DataFrame, cols:list[str]|str)->bool:
    """
    This function checks the viability of possible primary key
    Parameters
    ----------
    df : pl.DataFrame
        The dataframe to check
    cols : list[str]|str
        The columns to check
    """
    return len(df) == len(df.unique(cols))

# Pegar o dataset:
Para esse trabalho, foi escolhido o dataset olist_brazilian_ecommerce do kaggle.<br>
Esse dataset foi escolhido por conter multiplas tabelas relacionadas em um contexto de ecommerce brasileiro. Possibilitando uma análise mais complexa e um processo de pipeline mais robusto. 
- dataset name: Brazilian E-Commerce Public Dataset by Olist
- dataset link: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
- dataset licence: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

In [0]:
dataset_dir = "./dataset"
if not os.path.isdir(dataset_dir):
    os.mkdir(dataset_dir)

In [0]:
path = kagglehub.dataset_download(
    "olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)

In [0]:
os.listdir(path)

In [0]:
csv_files = [os.path.join(path, f) for f in os.listdir(path) if os.path.isfile(os.path.join(path, f)) and f.endswith(".csv")]
len(csv_files)

In [0]:
from shutil import copy2

for file in csv_files:
    dst_path = os.path.join(
        dataset_dir,
        os.path.split(file)[-1]
    )
    copy2(src=file, dst=dst_path)

In [0]:
customer_df = pl.read_csv(os.path.join(dataset_dir, 'olist_customers_dataset.csv'))
orders_df = pl.read_csv(os.path.join(dataset_dir,'olist_orders_dataset.csv'))
geolocation_df = pl.read_csv(os.path.join(dataset_dir,'olist_geolocation_dataset.csv'))
items_df = pl.read_csv(os.path.join(dataset_dir,'olist_order_items_dataset.csv'))
payments_df = pl.read_csv(os.path.join(dataset_dir,'olist_order_payments_dataset.csv'))
reviews_df = pl.read_csv(os.path.join(dataset_dir,'olist_order_reviews_dataset.csv'))
products_df = pl.read_csv(os.path.join(dataset_dir,'olist_products_dataset.csv'))
sellers_df = pl.read_csv(os.path.join(dataset_dir,'olist_sellers_dataset.csv'))
category_translation_df = pl.read_csv(os.path.join(dataset_dir,'product_category_name_translation.csv'))

# Análise exploratória:
Essa etapa faz uma analise simples de cada tabela, utilizando o método describe para descrever cada coluna e facilitar a construção de um modelo de entidade-relacionamento.<br>
Essa etapa **Não é obrigatoria para o entendimento do trabalho final**, porem ela ajuda na construção desse modelo e no entendimento dos dados a relação deles.

## CUSTOMERS_DATASET:

- customer_df:
    - <u>customer_id</u>: char(32), 
    - customer_unique_id: char(32), 
    - customer_zip_code_prefix: float
    - customer_city: varchar(50),
    - customer_state: char(2)
        

In [0]:
customer_df.describe()


In [0]:
customer_df.head()

In [0]:
for col in customer_df.schema:
    if customer_df.schema[col] == pl.String:
        print(max(customer_df.with_columns(len_col=pl.col(col).str.len_chars())["len_col"]))

In [0]:
# check primary key:
check_primary_key(customer_df, "customer_id")


## ORDER_DATASET

- orders_df:
    - <u>order_id</u>: char(32)
    - customer_id: char(32) -> FK.customer_df.customer_id
    - order_status: varchar(20)
    - order_purchase_timestamp: timestamp
    - order_aproved_at: timestamp
    - order_delivered_carrier_date: timestamp
    - order_delivered_customer_date: timestamp
    - order_estimated_delivery_date: date

In [0]:
orders_df.describe()

In [0]:
orders_df.group_by("order_status").len().with_columns(prop=(pl.col("len")/len(orders_df)).round(4)).sort("order_status")

In [0]:
# check primary key:
check_primary_key(orders_df, "order_id")

## GEOLOCATION_DATASET

- geolocation_df:
    - geolocation_zip_code_prefix: integer
    - geolocation_lat: decimal(8, 6)
    - geolocation_lng: decimal(9, 6)
    - geolocation_city: varchar(50)
    - geolocation_state: char(2)

In [0]:
geolocation_df.describe()

In [0]:
# Check if geolocation_zip_code_prefix is float or int
if len(geolocation_df.filter(pl.col("geolocation_zip_code_prefix") % 1 != 0)) == 0:
    print("Integer")
else:
    print("Float")

In [0]:
# check if any columns is unique:
for col in geolocation_df.columns:
    print(f"{col} -> is_unique: {check_primary_key(geolocation_df, col)}")


In [0]:
# Check possible pk:
check_primary_key(geolocation_df, ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"])

## ORDER ITEMS DATASET:

- items_df:
    - <u>order_id</u>: char(32) -> FK payment_df.order_id
    - <u>order_item_id</u>: int
    - product_id: char(32) -> FK product_df.product_id
    - seller_id : char(32) -> FK sellers_df.seller_id
    - shipping_limit_date: timestamp
    - price: decimal(10, 2)
    - freight_value: decimal(10, 2)

In [0]:
items_df.describe()

In [0]:
check_primary_key(items_df, ["order_id", "order_item_id"])

## ORDER PAYMENTS DATASET:

- payments_df:
    - <u>order_id</u>: char(32)
    - <u>payment_sequential</u>: int
    - payment_type: varchar(50)
    - payment_installments: int
    - payment_value: decimal(10, 2)

In [0]:
payments_df.describe()

In [0]:
check_primary_key(payments_df, ["order_id", "payment_sequential"])

## REVIEWS DATASET

- reviews_df:
    - <u>review_id</u>: char(32)
    - <u>order_id</u>: char(32) -> FK order_df.order_id
    - review_score: int
    - review_comment_title: varchar(255) CHARACTER SET utf8mb4
    - review_comment_message: varchar(2000) CHARACTER SET utf8mb4
    - review_creation_date: timestamp
    - review_answer_timestamp: timestamp

In [0]:
reviews_df.describe()

In [0]:
# Check primary_key
check_primary_key(reviews_df, ["review_id", "order_id"])

In [0]:
len(reviews_df.filter(pl.col("review_score")%1!=0))

## PRODUCT DATASET

- product_df:
    - <u>product_id</u>: char(32)
    - product_category_name: varchar(200)
    - product_name_lenght: int
    - product_description_lenght: int 
    - product_photos_qty: int
    - product_weight_g: decimal(8, 2)
    - product_lenght_cm: decimal(8,2)
    - product_heigh_cm: decimal(8,2)
    - product_width_cm: decimal(8,2)


In [0]:
products_df.describe()

In [0]:
check_primary_key(products_df, "product_id")

## SELLERS DATASET

- sellers_df:
    - <u>seller_id</u>: char(32)
    - seller_zip_code_prefix: int
    - seller_city: varchar(200) -> fix wrong value by geolocation zip code unique
    - seller_state: char(2)

In [0]:
sellers_df.describe()

In [0]:
check_primary_key(sellers_df, "seller_id")

In [0]:
print(
    sellers_df.filter(pl.col("seller_city").str.to_integer(strict=False).is_not_null())["seller_id"][0]
)
sellers_df.filter(pl.col("seller_city").str.to_integer(strict=False).is_not_null())

In [0]:
geolocation_df.filter(pl.col("geolocation_zip_code_prefix")==22790).unique("geolocation_city")

In [0]:
sellers_df = sellers_df.with_columns(
    seller_city = pl.when(pl.col("seller_city")=="04482255").then(
        pl.lit("rio de janeiro")
    ).otherwise(pl.col("seller_city"))
)
sellers_df.filter(pl.col("seller_id").str.contains("ceb7b4fb9401cd378de7886317ad1b47"))

## CATEGORY NAME TRANSLATION DATASET:

- category_translation:
    - <u>product_category_name</u>: varchar(200) -> FK product_df.product_category_name
    - product_category_name_english: varchar(200)

In [0]:
category_translation_df.describe()

In [0]:
check_primary_key(category_translation_df, "product_category_name")

# Modelo ER:
com base nessa análise, foi possível construir o seguinte diagrama utilizando o drawsql.app:
![raw dataset ER schema](./img/raw_er_schema.jpg)